# 08 — Component Mode Synthesis (CMS) Reduction

**Topic**: Craig-Bampton reduction — full vs. reduced eigenfrequencies, error vs. n_modes table.

**Reference**: Craig & Bampton (1968); Krack & Gross (2019) §6

**Estimated runtime**: < 20 seconds

## Theory

**Craig-Bampton (CB) reduction** (Craig & Bampton 1968) partitions the DOFs into *boundary* (b, interface) and *internal* (i) subsets, then builds the Ritz basis:

$$T = \begin{bmatrix} I_b & 0 \\ \Phi_c & \Phi_n \end{bmatrix} \tag{1}$$

where:
- $\Phi_c = -K_{ii}^{-1}K_{ib}$ — **constraint modes** (static response to unit boundary displacements)
- $\Phi_n$ — first $n_{int}$ **fixed-interface normal modes** (eigenmodes with boundary DOFs clamped)

The reduced-order model (ROM) has $n_b + n_{int}$ DOFs vs. the full model's $n_{full}$ DOFs:

$$M_r = T^\top M T, \quad K_r = T^\top K T \tag{2}$$

**Convergence**: as $n_{int}$ increases, the ROM eigenfrequencies converge monotonically to the full-order values from above (Rayleigh-Ritz upper bound property).

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

from nlvib.systems.fe_beam import FE_EulerBernoulliBeam
from nlvib.systems.cms import craig_bampton

print("Imports OK")

In [ ]:
# Build 20-element FE beam (clamped-free)
n_elements = 20  # ← try changing this (20 gives good full-order accuracy)
L    = 1.0
E    = 2e11
I    = 1e-5
rho  = 7800.0
A    = 1e-3

beam = FE_EulerBernoulliBeam(
    n_elements=n_elements, L=L, E=E, I_area=I, rho=rho, A=A, bc='clamped-free'
)

print(f"Full-order model: {beam.n_dof} DOFs")

# Full-order eigenfrequencies
K_full = beam.K.toarray()
M_full = beam.M.toarray()
vals_full, _ = la.eigh(K_full, M_full)
freqs_full = np.sqrt(np.maximum(vals_full, 0.0))

print("Full-order eigenfrequencies (rad/s), first 5:")
for i, f in enumerate(freqs_full[:5]):
    print(f"  Mode {i+1}: {f:.4f}")

In [ ]:
# Craig-Bampton with varying number of internal modes
# Boundary DOFs: clamped end (DOFs 0 and 1 = transverse + rotation at node 0)
# For clamped-free beam, the clamped node has DOFs 0 (transverse) and 1 (rotation)
# We use the free-end node as interface (tip DOFs)
n_internal_modes_list = [1, 2, 3, 5, 10]  # ← try changing this

# Boundary DOFs: tip (last two DOFs — transverse and rotation at free end)
boundary_dofs = [beam.n_dof - 2, beam.n_dof - 1]
print(f"Boundary DOFs: {boundary_dofs} (free end, transverse + rotation)")

n_modes_check = 5  # compare first 5 eigenfrequencies
errors_table = np.zeros((len(n_internal_modes_list), n_modes_check))

for row_idx, n_int in enumerate(n_internal_modes_list):
    rom, T = craig_bampton(beam, boundary_dofs=boundary_dofs, n_internal_modes=n_int)
    K_r = rom.K.toarray()
    M_r = rom.M.toarray()
    vals_r, _ = la.eigh(K_r, M_r)
    freqs_r = np.sqrt(np.maximum(vals_r, 0.0))

    for m_idx in range(n_modes_check):
        if m_idx < len(freqs_r):
            rel_err = abs(freqs_r[m_idx] - freqs_full[m_idx]) / freqs_full[m_idx]
            errors_table[row_idx, m_idx] = rel_err
        else:
            errors_table[row_idx, m_idx] = np.nan

    print(f"  n_int={n_int:2d}: ROM={rom.n_dof} DOFs, "
          f"freqs={np.round(freqs_r[:5], 2)}")

In [ ]:
# Print error table
print("\nRelative eigenfrequency errors (Craig-Bampton reduction):")
print(f"{'n_int':>6} | " + " | ".join([f'Mode {m+1:>7}' for m in range(n_modes_check)]))
print("-" * (6 + 13*n_modes_check))
for row_idx, n_int in enumerate(n_internal_modes_list):
    row = f"{n_int:>6} | "
    for m_idx in range(n_modes_check):
        err = errors_table[row_idx, m_idx]
        row += f"{err:>12.3e} | " if not np.isnan(err) else f"{'N/A':>12} | "
    print(row)

# Plot error vs n_modes
fig, ax = plt.subplots(figsize=(8, 4))
for m_idx in range(n_modes_check):
    valid = ~np.isnan(errors_table[:, m_idx])
    ax.semilogy(
        np.array(n_internal_modes_list)[valid],
        errors_table[valid, m_idx],
        'o-', lw=2, label=f'Mode {m_idx+1}'
    )

ax.set_xlabel('Number of internal modes $n_{int}$')
ax.set_ylabel('Relative eigenfrequency error')
ax.set_title('Craig-Bampton convergence — 20-element FE beam')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig

In [ ]:
# Compare full vs ROM mode shapes for mode 1
# Transformation: q_full ≈ T @ q_reduced
n_int_best = 5  # ← try changing this
rom_best, T_best = craig_bampton(beam, boundary_dofs=boundary_dofs, n_internal_modes=n_int_best)

K_rb = rom_best.K.toarray()
M_rb = rom_best.M.toarray()
vals_rb, vecs_rb = la.eigh(K_rb, M_rb)

# Full-order mode 1
_, vecs_full = la.eigh(K_full, M_full)

# ROM mode 1 expanded to full DOFs
q_rom_mode1_full = T_best @ vecs_rb[:, 0]

# For clamped-free beam with n_elements elements:
# beam.n_dof = 2 * n_elements  (clamped end's 2 DOFs are removed, free end has 2 DOFs)
# DOFs are ordered: [trans_1, rot_1, trans_2, rot_2, ..., trans_n, rot_n]
# corresponding to nodes 1..n (node 0 is clamped/fixed)
# Extract transverse DOFs (even indices: 0,2,4,...)
trans_dofs = np.arange(0, beam.n_dof, 2)                  # 20 values for n_elements=20
nodes_x = np.linspace(0, L, n_elements + 1)[1:]           # skip clamped node at x=0

q_full_mode1 = vecs_full[:, 0][trans_dofs]
q_rom_mode1  = q_rom_mode1_full[trans_dofs]

# Prepend clamped-end (zero displacement at x=0)
nodes_x_plot   = np.concatenate([[0.0], nodes_x])
q_full_plot    = np.concatenate([[0.0], q_full_mode1])
q_rom_plot     = np.concatenate([[0.0], q_rom_mode1])

# Normalise to same sign
q_full_plot /= np.max(np.abs(q_full_plot)) + 1e-15
q_rom_plot  /= np.max(np.abs(q_rom_plot))  + 1e-15
if np.sign(q_full_plot[-1]) != np.sign(q_rom_plot[-1]):
    q_rom_plot *= -1

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nodes_x_plot, q_full_plot, 'o-', lw=2, label='Full-order (20 elements)', color='tab:blue')
ax.plot(nodes_x_plot, q_rom_plot,  's--', lw=2, label=f'Craig-Bampton ROM (n_int={n_int_best})', color='tab:orange')
ax.set_xlabel('Position $x$ (m)')
ax.set_ylabel('Normalised displacement')
ax.set_title('Mode 1 shape: full-order vs Craig-Bampton ROM')
ax.legend()
ax.grid(True, alpha=0.3)
fig

## Key Takeaways

- Craig-Bampton reduces a 40-DOF beam to as few as 4 DOFs while retaining the first mode accurately.
- Error decreases monotonically with $n_{int}$ — the ROM gives an upper bound on each eigenfrequency.
- Low modes converge faster: mode 1 is accurate even with $n_{int}=1$; mode 5 needs $n_{int}\geq 5$.
- The mode shape from the ROM (expanded via $T$) visually matches the full-order shape for $n_{int}\geq 3$.